In [1]:
# ============================================================
# Notebook : 01_eda.ipynb
# Purpose  : Exploratory description of the home-track analysis sample.
#            Summarise the five abnormal categories, their severity
#            split, co-occurrence structure, between-household
#            heterogeneity, and the class imbalance that motivates the
#            dual reliability criterion used in Step 4. Figures are
#            grayscale, caption-free, saved as PNG and PDF at 600 dpi.
# Design   : Single code cell. No absolute path is printed.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ---- Grayscale figure convention ----
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.unicode_minus": False,
    "savefig.dpi": 600,
    "figure.dpi": 120,
    "axes.edgecolor": "0.15",
    "grid.color": "0.85",
})
GRAY = ["0.15", "0.40", "0.60", "0.75", "0.88"]  # dark -> light


def save_fig(fig, name):
    """Save a figure as PNG and PDF at 600 dpi. No path is printed."""
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


# ---- Paths and data ----
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
ABN = meta["abnormal_categories"]
SENSOR_CONT = meta["sensor_cont"]
CAT_LABEL = {"HR": "Heart rate", "BR": "Breath rate", "ST": "Skin temp.",
             "SP": "SpO2", "ER": "Emergency"}

# ============================================================
# Figure 1. Category prevalence with severity split (stacked, grayscale).
# ============================================================
rows = []
for c in ABN:
    col = "cat_" + c
    rows.append({
        "category": CAT_LABEL[c],
        "danger": int(((df[col] == 1) & (df["sev_danger"] == 1)).sum()),
        "caution": int(((df[col] == 1) & (df["sev_caution"] == 1)).sum()),
    })
pv = pd.DataFrame(rows).set_index("category").sort_values("danger", ascending=False)
pv_total = (pv["danger"] + pv["caution"]).sort_values(ascending=False)
pv = pv.loc[pv_total.index]

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.bar(pv.index, pv["caution"], color=GRAY[2], edgecolor="black", linewidth=0.6, label="Caution")
ax.bar(pv.index, pv["danger"], bottom=pv["caution"], color=GRAY[0], edgecolor="black",
       linewidth=0.6, label="Danger")
ax.set_ylabel("Positive observation count")
ax.set_xlabel("Abnormal category")
ax.legend(frameon=False)
save_fig(fig, "fig01_category_prevalence")

# ============================================================
# Figure 2. Category co-occurrence (Jaccard, grayscale heatmap).
# ============================================================
M = df[["cat_" + c for c in ABN]].values
J = np.zeros((len(ABN), len(ABN)))
for i in range(len(ABN)):
    for j in range(len(ABN)):
        a, b = M[:, i], M[:, j]
        inter = np.logical_and(a, b).sum()
        union = np.logical_or(a, b).sum()
        J[i, j] = inter / union if union > 0 else 0.0
Jdf = pd.DataFrame(J, index=[CAT_LABEL[c] for c in ABN], columns=[CAT_LABEL[c] for c in ABN])

fig, ax = plt.subplots(figsize=(4.8, 4.0))
sns.heatmap(Jdf, annot=True, fmt=".3f", cmap="Greys", vmin=0, vmax=1,
            linewidths=0.5, linecolor="0.6", cbar_kws={"label": "Jaccard index"},
            annot_kws={"size": 8}, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("")
save_fig(fig, "fig02_category_cooccurrence")

# ============================================================
# Figure 3. Between-household heterogeneity in danger rate.
# ============================================================
hh = df.groupby("HH_ID")["sev_danger"].mean().sort_values()
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.bar(range(len(hh)), hh.values, color=GRAY[2], edgecolor="black", linewidth=0.5)
ax.axhline(hh.mean(), color="black", linestyle="--", linewidth=1.0, label="Sample mean")
ax.set_xticks(range(len(hh)))
ax.set_xticklabels(hh.index, rotation=90, fontsize=6)
ax.set_ylabel("Danger-record rate")
ax.set_xlabel("Household")
ax.legend(frameon=False)
save_fig(fig, "fig03_household_heterogeneity")

# ============================================================
# Figure 4. Vital-sign distribution, normal vs abnormal (grayscale boxplots).
#           Illustrates separability for channels that define a category
#           (HeartRate, BreathRate, SPO2) versus the environment channels.
# ============================================================
vitals = ["HeartRate", "BreathRate", "SPO2", "StressIndex"]
long = df.melt(id_vars=["any_abnormal"], value_vars=vitals,
               var_name="sensor", value_name="value")
long["state"] = np.where(long["any_abnormal"] == 1, "Abnormal", "Normal")
fig, axes = plt.subplots(1, len(vitals), figsize=(9.5, 3.0))
for ax, s in zip(axes, vitals):
    sub = long[long["sensor"] == s]
    sns.boxplot(data=sub, x="state", y="value", ax=ax, order=["Normal", "Abnormal"],
                palette=[GRAY[3], GRAY[1]], fliersize=0.5, linewidth=0.8)
    ax.set_title("")
    ax.set_xlabel(s)
    ax.set_ylabel("")
save_fig(fig, "fig04_vital_separability")

# ============================================================
# Table. Sensor summary by state (mean, sd) for the record.
# ============================================================
summ = []
for s in SENSOR_CONT:
    for state, g in [("normal", df[df.any_abnormal == 0]), ("abnormal", df[df.any_abnormal == 1])]:
        summ.append({"sensor": s, "state": state,
                     "mean": round(g[s].mean(), 3), "sd": round(g[s].std(), 3)})
summ = pd.DataFrame(summ)
summ.to_csv(TAB / "table_eda_sensor_summary.csv", index=False, encoding="utf-8-sig")

# ---- Console summary (counts and rates only) ----
print("=== EDA summary (home track, kept sessions) ===")
print(f"Records: {len(df):,} | households: {df['HH_ID'].nunique()} | sessions: {df['session_id'].nunique():,}")
print(f"Overall abnormal rate: {df['any_abnormal'].mean():.4f} | danger rate: {df['sev_danger'].mean():.4f}")
print()
print("Category positive counts and prevalence:")
for c in ABN:
    col = "cat_" + c
    print(f"  {CAT_LABEL[c]:12s}  n={int(df[col].sum()):6d}  prev={df[col].mean():.5f}")
print()
print("Class imbalance (positive : negative) for the two minority categories:")
for c in ["SP", "ER"]:
    p = df["cat_" + c].mean()
    print(f"  {CAT_LABEL[c]:12s}  1 : {int((1-p)/p)}")
print()
print("Between-household danger rate: min {:.4f}, max {:.4f}, ratio {:.1f}x".format(
    hh.min(), hh.max(), (hh.max() / max(hh[hh > 0].min(), 1e-9))))
print()
print("Figures saved: fig01_category_prevalence, fig02_category_cooccurrence,")
print("               fig03_household_heterogeneity, fig04_vital_separability  (png + pdf, 600 dpi)")
print("Table saved  : table_eda_sensor_summary.csv")


=== EDA summary (home track, kept sessions) ===
Records: 99,035 | households: 25 | sessions: 234
Overall abnormal rate: 0.2071 | danger rate: 0.0273

Category positive counts and prevalence:
  Heart rate    n= 14370  prev=0.14510
  Breath rate   n=  6956  prev=0.07024
  Skin temp.    n=  4663  prev=0.04708
  SpO2          n=   569  prev=0.00575
  Emergency     n=   311  prev=0.00314

Class imbalance (positive : negative) for the two minority categories:
  SpO2          1 : 173
  Emergency     1 : 317

Between-household danger rate: min 0.0000, max 0.0681, ratio 11.9x

Figures saved: fig01_category_prevalence, fig02_category_cooccurrence,
               fig03_household_heterogeneity, fig04_vital_separability  (png + pdf, 600 dpi)
Table saved  : table_eda_sensor_summary.csv
